# Water Quality Prediction — Milestone 1: Setup & Dataset Loading

**Dataset:** `water_potability.csv` — 3,276 water samples, each labelled 0 (unsafe) or 1 (safe)  
**Goal of this notebook:** confirm the environment works, load the dataset, understand its shape and quirks before we touch anything.

> Think of this notebook like the first commit on a new project — get the scaffolding right before writing any real logic.

## Cell 1 — Imports

Pulling in everything we'll need across all milestones. Matplotlib gets a non-interactive backend so figures render cleanly in notebooks.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless rendering — figures saved to disk, not popped up
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import joblib
import os

print('pandas', pd.__version__)
print('numpy', np.__version__)
print('sklearn and xgboost loaded')
print('\n✅ T1.1 PASS — all imports OK')

pandas 2.3.3
numpy 2.0.2
sklearn and xgboost loaded

✅ T1.1 PASS — all imports OK


## Cell 2 — Load Dataset

Reading from the `data/` folder. The CSV has 9 numeric feature columns and one binary target (`Potability`).  
We'll name the DataFrame `water_df` throughout — more specific than a plain `df` and makes the domain obvious when you're deep in preprocessing cells.

In [2]:
water_df = pd.read_csv('../data/water_potability.csv')

print(f'Shape: {water_df.shape}')
print(f'Columns: {list(water_df.columns)}')
print()
water_df.head()

Shape: (3276, 10)
Columns: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']



,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


## Cell 3 — Basic Dataset Statistics

`.describe()` is the equivalent of a quick `SELECT MIN, MAX, AVG, STDDEV` on every column.  
Notice that `ph`, `Sulfate`, and `Trihalomethanes` will show a `count` below 3276 — that's where the nulls are hiding.

In [3]:
print('=== Descriptive Statistics ===')
display(water_df.describe().round(3))

print('\n=== Data Types ===')
print(water_df.dtypes)

print('\n=== Missing Values per Column ===')
missing = water_df.isnull().sum()
print(missing[missing > 0])  # only show columns that actually have nulls

total_missing = missing.sum()
pct_missing = (total_missing / water_df.size) * 100
print(f'\nTotal missing cells: {total_missing} ({pct_missing:.1f}% of dataset)')

=== Descriptive Statistics ===


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
count,2785.000,3276.000,3276.000,3276.000,2495.000,3276.000,3276.000,3114.000,3276.000,3276.000
mean,7.081,196.369,22014.093,7.122,333.776,426.205,14.285,66.396,3.967,0.390
std,1.594,32.880,8768.571,1.583,41.417,80.824,3.308,16.175,0.780,0.488
min,0.000,47.432,320.943,0.352,129.000,181.484,2.200,0.738,1.450,0.000
25%,6.093,176.851,15666.690,6.127,307.699,365.734,12.066,55.845,3.440,0.000
50%,7.037,196.968,20927.834,7.130,333.074,421.885,14.218,66.622,3.955,0.000
75%,8.062,216.667,27332.762,8.115,359.950,481.792,16.558,77.337,4.500,1.000
max,14.000,323.124,61227.196,13.127,481.031,753.343,28.300,124.000,6.739,1.000



=== Data Types ===
ph                 float64
Hardness           float64
Solids             float64
Chloramines        float64
Sulfate            float64
Conductivity       float64
Organic_carbon     float64
Trihalomethanes    float64
Turbidity          float64
Potability           int64
dtype: object

=== Missing Values per Column ===
ph                 491
Sulfate            781
Trihalomethanes    162
dtype: int64

Total missing cells: 1434 (4.4% of dataset)


## Cell 4 — Class Distribution

The target is binary — 0 means unsafe, 1 means safe.  
This dataset is not balanced: ~61% unsafe, ~39% safe.  
That matters later: a model that just always predicts 'unsafe' gets 61% accuracy without learning anything. This is why we'll use F1-score and AUC alongside accuracy when evaluating models.

In [4]:
class_counts = water_df['Potability'].value_counts()
safe_pct = class_counts[1] / len(water_df) * 100
unsafe_pct = class_counts[0] / len(water_df) * 100

print('Class Distribution:')
print(f'  Not Safe (0): {class_counts[0]} samples  ({unsafe_pct:.1f}%)')
print(f'  Safe     (1): {class_counts[1]} samples  ({safe_pct:.1f}%)')
print()
print('⚠️  Imbalance noted — accuracy alone will be misleading. F1 and AUC are the real metrics.')

Class Distribution:
  Not Safe (0): 1998 samples  (61.0%)
  Safe     (1): 1278 samples  (39.0%)

⚠️  Imbalance noted — accuracy alone will be misleading. F1 and AUC are the real metrics.


## Cell 5 — Column Overview

Quick check on value ranges for each feature. Useful sanity check — if pH goes above 14 or below 0 those are measurement errors, not real data.

In [5]:
feature_cols = [c for c in water_df.columns if c != 'Potability']

header = '{:<22} {:>10} {:>10} {:>10} {:>8}'.format('Feature', 'Min', 'Max', 'Mean', 'Nulls')
print(header)
print('-' * 65)
for col in feature_cols:
    nulls = water_df[col].isnull().sum()
    null_marker = '{} (missing)'.format(nulls) if nulls > 0 else str(nulls)
    row = '{:<22} {:>10.2f} {:>10.2f} {:>10.2f} {:>14}'.format(
        col, water_df[col].min(), water_df[col].max(), water_df[col].mean(), null_marker
    )
    print(row)

Feature                       Min        Max       Mean    Nulls
-----------------------------------------------------------------
ph                           0.00      14.00       7.08  491 (missing)
Hardness                    47.43     323.12     196.37              0
Solids                     320.94   61227.20   22014.09              0
Chloramines                  0.35      13.13       7.12              0
Sulfate                    129.00     481.03     333.78  781 (missing)
Conductivity               181.48     753.34     426.21              0
Organic_carbon               2.20      28.30      14.28              0
Trihalomethanes              0.74     124.00      66.40  162 (missing)
Turbidity                    1.45       6.74       3.97              0


---

## Milestone 1 Tests

Run these cells to confirm M1 is complete. A clean run with no `AssertionError` = milestone passed.

> Same discipline as not merging a PR until CI is green.

In [6]:
# T1.1 — already ran at the top of this notebook (import cell)
# Re-asserting here so all tests are in one place

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import joblib

print('✅ T1.1 PASS — all imports OK')

✅ T1.1 PASS — all imports OK


In [7]:
# T1.2 — dataset loads with correct shape
water_df = pd.read_csv('../data/water_potability.csv')

assert water_df.shape[1] == 10, f"Expected 10 columns, got {water_df.shape[1]}"
assert water_df.shape[0] > 3000, f"Expected 3000+ rows, got {water_df.shape[0]}"
assert 'Potability' in water_df.columns, "Target column 'Potability' is missing"

print(f'✅ T1.2 PASS — dataset shape: {water_df.shape}')

✅ T1.2 PASS — dataset shape: (3276, 10)


In [8]:
# T1.3 — target column is strictly binary
unique_vals = set(water_df['Potability'].unique())

assert unique_vals == {0, 1}, f"Expected only 0 and 1, got: {unique_vals}"

print(f'✅ T1.3 PASS — Potability values: {unique_vals}')

✅ T1.3 PASS — Potability values: {np.int64(0), np.int64(1)}


In [9]:
# T1.4 — folder structure is in place
import os

# paths are relative to the project root, not notebooks/
required_dirs = ['../data', '../notebooks', '../models', '../reports/figures', '../docs']
for folder in required_dirs:
    assert os.path.exists(folder), f"Missing folder: {folder}"

print('✅ T1.4 PASS — folder structure is in place')

✅ T1.4 PASS — folder structure is in place


In [10]:
# Final summary — what we know heading into M2
print('=== Milestone 1 Complete ===')
print(f'Dataset shape:   {water_df.shape}')
print(f'Features:        {[c for c in water_df.columns if c != "Potability"]}')
print(f'Target:          Potability (0=unsafe, 1=safe)')
print(f'Missing values:  ph={water_df["ph"].isnull().sum()}, Sulfate={water_df["Sulfate"].isnull().sum()}, Trihalomethanes={water_df["Trihalomethanes"].isnull().sum()}')
print(f'Class split:     {water_df["Potability"].value_counts()[0]} unsafe / {water_df["Potability"].value_counts()[1]} safe')
print()
print('Next: Milestone 2 — EDA (distributions, heatmap, boxplots, class imbalance)')

=== Milestone 1 Complete ===
Dataset shape:   (3276, 10)
Features:        ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']
Target:          Potability (0=unsafe, 1=safe)
Missing values:  ph=491, Sulfate=781, Trihalomethanes=162
Class split:     1998 unsafe / 1278 safe

Next: Milestone 2 — EDA (distributions, heatmap, boxplots, class imbalance)


---

## Milestone 2 — Exploratory Data Analysis

Understanding the data before building models is non-negotiable.  
The patterns we find here drive every preprocessing and modelling decision downstream.

**Figures produced by this section:**
- Figure 2.1 — Feature distributions (safe vs unsafe overlaid)
- Figure 2.2 — Class distribution (the imbalance we need to account for)
- Figure 2.3 — Correlation matrix heatmap
- Figure 2.4 — Boxplots by potability class
- Figure 2.5 — Violin plots (full distribution shape per class)

In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

water_df = pd.read_csv('../data/water_potability.csv')
feature_cols = [c for c in water_df.columns if c != 'Potability']

os.makedirs('../reports/figures', exist_ok=True)

# consistent colours throughout all EDA figures so the legend is always readable
SAFE_COLOR   = '#4c9be8'
UNSAFE_COLOR = '#e07b54'

print(f'{len(feature_cols)} features, {len(water_df)} samples, {water_df.isnull().sum().sum()} missing cells total')

9 features, 3276 samples, 1434 missing cells total


### Figure 2.1 — Feature Distributions

Overlaying safe vs unsafe on the same histogram immediately shows whether a feature separates the two classes.  
If the two bars look identical, that feature probably isn't very predictive — a hint about feature importance later.

In [12]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes_flat = axes.flatten()

for i, col in enumerate(feature_cols):
    safe_vals   = water_df.loc[water_df['Potability'] == 1, col].dropna()
    unsafe_vals = water_df.loc[water_df['Potability'] == 0, col].dropna()

    axes_flat[i].hist(unsafe_vals, bins=30, alpha=0.55, color=UNSAFE_COLOR,
                      label='Not Safe', density=True, edgecolor='none')
    axes_flat[i].hist(safe_vals,   bins=30, alpha=0.55, color=SAFE_COLOR,
                      label='Safe',     density=True, edgecolor='none')
    axes_flat[i].set_title(col, fontsize=11, fontweight='bold')
    axes_flat[i].set_xlabel('Value', fontsize=9)
    axes_flat[i].set_ylabel('Density', fontsize=9)
    axes_flat[i].legend(fontsize=8, framealpha=0.7)

plt.suptitle('Figure 2.1: Feature Distributions by Potability Class',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/fig2_1_distributions.png', bbox_inches='tight', dpi=150)
plt.close()
print('Figure 2.1 saved.')
print('Observation: safe and unsafe distributions overlap heavily — no single feature cleanly separates the classes.')
print('This explains why the ceiling accuracy on this dataset is ~70%, not ~95%.')

Figure 2.1 saved.
Observation: safe and unsafe distributions overlap heavily — no single feature cleanly separates the classes.
This explains why the ceiling accuracy on this dataset is ~70%, not ~95%.


### Figure 2.2 — Class Distribution

The dataset is not balanced. A naïve model that predicts "unsafe" for every sample would hit ~61% accuracy — decent-looking but completely useless.  
This is why we will use **F1-score and ROC-AUC** as primary metrics, not accuracy.

In [13]:
counts = water_df['Potability'].value_counts().sort_index()

fig, (ax_bar, ax_pie) = plt.subplots(1, 2, figsize=(11, 5))

bars = ax_bar.bar(['Not Safe (0)', 'Safe (1)'], [counts[0], counts[1]],
                  color=[UNSAFE_COLOR, SAFE_COLOR], edgecolor='white', linewidth=1.5, width=0.5)
for bar, count in zip(bars, [counts[0], counts[1]]):
    pct = count / len(water_df) * 100
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 25,
                '{}\n({:.1f}%)'.format(count, pct),
                ha='center', va='bottom', fontweight='bold', fontsize=11)
ax_bar.set_title('Sample Counts', fontsize=12, fontweight='bold')
ax_bar.set_ylabel('Number of Samples', fontsize=10)
ax_bar.set_ylim(0, counts[0] * 1.25)
ax_bar.spines[['top', 'right']].set_visible(False)

wedges, texts, autotexts = ax_pie.pie(
    [counts[0], counts[1]],
    labels=['Not Safe (0)', 'Safe (1)'],
    colors=[UNSAFE_COLOR, SAFE_COLOR],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontweight('bold')
ax_pie.set_title('Class Proportion', fontsize=12, fontweight='bold')

plt.suptitle('Figure 2.2: Class Distribution — 61% Unsafe vs 39% Safe',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/fig2_2_class_dist.png', bbox_inches='tight', dpi=150)
plt.close()
print('Figure 2.2 saved.')
print(f'Class split: {counts[0]} unsafe ({counts[0]/len(water_df)*100:.1f}%) | {counts[1]} safe ({counts[1]/len(water_df)*100:.1f}%)')

Figure 2.2 saved.
Class split: 1998 unsafe (61.0%) | 1278 safe (39.0%)


### Figure 2.3 — Correlation Heatmap

Low correlations between features means we keep all of them — no redundancy to drop.  
Low correlation with `Potability` doesn't mean a feature is useless — tree-based models capture non-linear relationships that this linear measure misses entirely.

In [14]:
corr_matrix = water_df.corr(numeric_only=True)

# only show the lower triangle — the upper is a mirror image, showing both just adds noise
upper_mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr_matrix,
    mask=upper_mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 9},
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Figure 2.3: Feature Correlation Matrix (lower triangle)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../reports/figures/fig2_3_heatmap.png', bbox_inches='tight', dpi=150)
plt.close()
print('Figure 2.3 saved.')

potability_corr = corr_matrix['Potability'].drop('Potability').abs().sort_values(ascending=False)
print('\nCorrelation with Potability (absolute values):')
print(potability_corr.to_string())
print('\nNote: all are weak (<0.1). Non-linear models (RF, XGBoost) will handle this better than Logistic Regression.')

Figure 2.3 saved.

Correlation with Potability (absolute values):
Solids             0.033743
Organic_carbon     0.030001
Chloramines        0.023779
Sulfate            0.023577
Hardness           0.013837
Conductivity       0.008128
Trihalomethanes    0.007130
ph                 0.003556
Turbidity          0.001581

Note: all are weak (<0.1). Non-linear models (RF, XGBoost) will handle this better than Logistic Regression.


### Figure 2.4 — Boxplots by Potability Class

Boxplots show median, spread, and outlier count for each feature split by class.  
Look for features where the boxes are clearly offset — those carry the most discriminating power.  
Heavy overlap = model needs to combine multiple features to predict class, not just one.

In [15]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes_flat = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.boxplot(
        data=water_df,
        x='Potability',
        y=col,
        hue='Potability',
        palette={0: UNSAFE_COLOR, 1: SAFE_COLOR},
        ax=axes_flat[i],
        width=0.5,
        flierprops={'marker': 'o', 'markersize': 2, 'alpha': 0.35},
        legend=False
    )
    axes_flat[i].set_title(col, fontsize=11, fontweight='bold')
    axes_flat[i].set_xlabel('Potability', fontsize=9)
    axes_flat[i].set_ylabel('Value', fontsize=9)
    axes_flat[i].set_xticks([0, 1])
    axes_flat[i].set_xticklabels(['Not Safe', 'Safe'], fontsize=9)

plt.suptitle('Figure 2.4: Feature Distributions by Potability Class (Boxplots)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/fig2_4_boxplots.png', bbox_inches='tight', dpi=150)
plt.close()
print('Figure 2.4 saved.')
print('Observation: most features show similar medians and IQR for both classes — the classes are genuinely hard to separate.')

Figure 2.4 saved.
Observation: most features show similar medians and IQR for both classes — the classes are genuinely hard to separate.


### IQR Outlier Analysis

Computing outlier counts per feature using the standard IQR fence: `Q1 - 1.5×IQR` to `Q3 + 1.5×IQR`.  
This confirms the decision to use **median imputation** in M3 — median is robust to outliers, mean is not.  
No feature has a rate severe enough to warrant row removal.

In [16]:
outlier_rows = []
for col in feature_cols:
    col_data = water_df[col].dropna()
    q1, q3   = col_data.quantile(0.25), col_data.quantile(0.75)
    iqr      = q3 - q1
    lo_fence = q1 - 1.5 * iqr
    hi_fence = q3 + 1.5 * iqr
    outliers = col_data[(col_data < lo_fence) | (col_data > hi_fence)]
    outlier_rows.append({
        'Feature': col,
        'IQR': round(iqr, 2),
        'Lower Fence': round(lo_fence, 2),
        'Upper Fence': round(hi_fence, 2),
        'Outlier Count': len(outliers),
        'Outlier %': round(len(outliers) / len(col_data) * 100, 1)
    })

outlier_df = pd.DataFrame(outlier_rows).set_index('Feature')
print('=== Outlier Analysis (IQR Method) ===')
display(outlier_df)

worst_col = outlier_df['Outlier %'].idxmax()
max_pct   = outlier_df['Outlier %'].max()
print(f'\nHighest outlier rate: {worst_col} at {max_pct:.1f}% — mild. All features stay in.')

=== Outlier Analysis (IQR Method) ===


,IQR,Lower Fence,Upper Fence,Outlier Count,Outlier %
Feature,,,,,
ph,1.97,3.14,11.02,46,1.7
Hardness,39.82,117.13,276.39,83,2.5
Solids,11666.07,-1832.42,44831.87,47,1.4
Chloramines,1.99,3.15,11.10,61,1.9
Sulfate,52.25,229.32,438.33,41,1.6
Conductivity,116.06,191.65,655.88,11,0.3
Organic_carbon,4.49,5.33,23.30,25,0.8
Trihalomethanes,21.49,23.61,109.58,33,1.1
Turbidity,1.06,1.85,6.09,19,0.6



Highest outlier rate: Hardness at 2.5% — mild. All features stay in.


### Figure 2.5 — Violin Plots

Violin plots show the **full distribution shape** via kernel density estimation — not just quartiles.  
They reveal bimodality and skewness that a boxplot hides, and make it clearer when two classes have different density peaks.

In [17]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes_flat = axes.flatten()

for i, col in enumerate(feature_cols):
    # dropna per column — each feature has different null positions
    col_data = water_df[['Potability', col]].dropna()
    sns.violinplot(
        data=col_data,
        x='Potability',
        y=col,
        hue='Potability',
        palette={0: UNSAFE_COLOR, 1: SAFE_COLOR},
        ax=axes_flat[i],
        inner='quartile',
        cut=0,
        legend=False
    )
    axes_flat[i].set_title(col, fontsize=11, fontweight='bold')
    axes_flat[i].set_xlabel('Potability', fontsize=9)
    axes_flat[i].set_ylabel('Value', fontsize=9)
    axes_flat[i].set_xticks([0, 1])
    axes_flat[i].set_xticklabels(['Not Safe', 'Safe'], fontsize=9)

plt.suptitle('Figure 2.5: Feature Distributions by Class (Violin Plots)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/fig2_5_violin_plots.png', bbox_inches='tight', dpi=150)
plt.close()
print('Figure 2.5 saved.')

Figure 2.5 saved.


### EDA Summary — Key Insights

**1. Class Imbalance**  
1,998 unsafe (61.0%) vs 1,278 safe (39.0%). Accuracy will be systematically inflated. F1-score and ROC-AUC are the metrics that tell the real story.

**2. Missing Values**  
Three columns have significant nulls: `ph` (491 = 15%), `Sulfate` (781 = 23.8%), `Trihalomethanes` (162 = 5.2%).  
Impute with column **median** — the IQR analysis shows mild outliers in every feature, and median is more robust than mean in this case.

**3. Feature Separability**  
All features show substantial overlap between safe and unsafe distributions (Figures 2.1 and 2.4). No single feature cleanly discriminates — the model needs to combine all 9.

**4. No Multicollinearity**  
No feature pair exceeds 0.85 correlation. Keep all 9 features — no redundancy to drop.

**5. Weak Linear Signal**  
All features have low (< 0.1) absolute correlation with `Potability`. Logistic Regression will underperform compared to tree-based models that capture non-linear interactions.

**6. Outliers**  
Mild outliers present in all features (Hardness worst at 2.5%). None severe enough to drop rows. Confirms median > mean for imputation.

In [18]:
# T2.1 — missing values identified
missing = water_df.isnull().sum()
cols_with_nulls = missing[missing > 0]

assert 'ph' in cols_with_nulls.index, "Expected nulls in 'ph'"
assert 'Sulfate' in cols_with_nulls.index, "Expected nulls in 'Sulfate'"
assert 'Trihalomethanes' in cols_with_nulls.index, "Expected nulls in 'Trihalomethanes'"

print('✅ T2.1 PASS — missing values identified:')
print(cols_with_nulls.to_string())

✅ T2.1 PASS — missing values identified:
ph                 491
Sulfate            781
Trihalomethanes    162


In [19]:
# T2.2 — class imbalance documented
counts   = water_df['Potability'].value_counts()
safe_pct = counts[1] / len(water_df) * 100

assert 30 < safe_pct < 50, f'Unexpected class ratio: {safe_pct:.1f}% safe'

print(f'✅ T2.2 PASS — class split: {safe_pct:.1f}% safe / {100 - safe_pct:.1f}% unsafe')
print('Note: this imbalance means accuracy alone is a misleading metric — use F1 and AUC')

✅ T2.2 PASS — class split: 39.0% safe / 61.0% unsafe
Note: this imbalance means accuracy alone is a misleading metric — use F1 and AUC


In [20]:
# T2.3 — all 4 required EDA figures saved to disk
import os

eda_figures = [
    '../reports/figures/fig2_1_distributions.png',
    '../reports/figures/fig2_2_class_dist.png',
    '../reports/figures/fig2_3_heatmap.png',
    '../reports/figures/fig2_4_boxplots.png',
]
for path in eda_figures:
    assert os.path.exists(path), f'Missing figure: {path}'
    assert os.path.getsize(path) > 10_000, f'Figure looks empty (too small): {path}'
    print('  ✓ {}'.format(os.path.basename(path)))

print('✅ T2.3 PASS — all 4 EDA figures saved')

  ✓ fig2_1_distributions.png
  ✓ fig2_2_class_dist.png
  ✓ fig2_3_heatmap.png
  ✓ fig2_4_boxplots.png
✅ T2.3 PASS — all 4 EDA figures saved


In [21]:
# T2.4 — no extreme multicollinearity between features
feature_only     = water_df.drop('Potability', axis=1)
corr_abs         = feature_only.corr(numeric_only=True).abs()
upper_tri        = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
high_corr_pairs  = upper_tri.stack()[upper_tri.stack() > 0.85]

assert len(high_corr_pairs) == 0, f'High correlation detected:\n{high_corr_pairs}'

print('✅ T2.4 PASS — no multicollinearity above 0.85 threshold')
print('All 9 features are independent enough to keep.')

✅ T2.4 PASS — no multicollinearity above 0.85 threshold
All 9 features are independent enough to keep.


In [22]:
print('=== Milestone 2 Complete ===')
print('Figures saved:  fig2_1 through fig2_5 in reports/figures/')
print('Missing values: ph=491, Sulfate=781, Trihalomethanes=162  →  impute with median in M3')
print('Class split:    61% unsafe / 39% safe  →  report F1 and AUC, not just accuracy')
print('Correlation:    no pair exceeds 0.85  →  keep all 9 features')
print()
print('Next: Milestone 3 — Preprocessing (imputation, train/test split, feature scaling)')

=== Milestone 2 Complete ===
Figures saved:  fig2_1 through fig2_5 in reports/figures/
Missing values: ph=491, Sulfate=781, Trihalomethanes=162  →  impute with median in M3
Class split:    61% unsafe / 39% safe  →  report F1 and AUC, not just accuracy
Correlation:    no pair exceeds 0.85  →  keep all 9 features

Next: Milestone 3 — Preprocessing (imputation, train/test split, feature scaling)
